In [16]:
import pandas as pd 
import recordlinkage as rl
df_spotify = pd.read_csv("../data/processed/SpotifyCleaned.csv")
df_million_song = pd.read_csv("../data/processed/MillionSongCleaned.csv")

In [17]:
# --- Create lowercase versions of the title/artist fields ---
# df_spotify["ArtistLower"] = df_spotify["artist"].str.lower()
# df_spotify["SongLower"] = df_spotify["song"].str.lower()

df_spotify["ArtistLower"] = df_spotify["artists"].str.lower()
df_spotify["SongLower"] = df_spotify["track_name"].str.lower()


df_million_song["ArtistLower"] = df_million_song["ArtistName"].str.lower()
df_million_song["SongLower"] = df_million_song["Title"].str.lower()

df_exact_join = pd.merge(
    df_spotify,
    df_million_song,
    left_on=['ArtistLower', 'SongLower'],  
    right_on=['ArtistLower', 'SongLower'], 
    how='inner'                        
)
print(len(df_exact_join))

100


In [18]:
spotify_idxs = df_exact_join.set_index(['ArtistLower', 'SongLower']).index
df_spotify_remaining = df_spotify[~df_spotify.set_index(['ArtistLower', 'SongLower']).index.isin(spotify_idxs)].copy()

million_idxs = df_exact_join.set_index(['ArtistLower', 'SongLower']).index
df_million_song_remaining = df_million_song[~df_million_song.set_index(['ArtistLower', 'SongLower']).index.isin(million_idxs)].copy()

In [19]:
print(len(df_spotify))
print(len(df_exact_join))
print(len(df_spotify_remaining))
print(len(df_million_song))
print(len(df_exact_join))
print(len(df_million_song_remaining))

76585
100
76485
9942
100
9843


In [26]:
def first_word(name):
    first = name.strip().split()[0].lower()
    if first == "the" and len(name.strip().split()) > 1:
        first = name.strip().split()[1].lower()
    return first

df_spotify_remaining["FirstWord"] = df_spotify_remaining["ArtistLower"].apply(first_word)
df_million_song_remaining["FirstWord"] = df_million_song_remaining["ArtistLower"].apply(first_word)


indexer = rl.Index()
indexer.block("FirstWord")
pairs = indexer.index(df_spotify_remaining, df_million_song_remaining)


def get_matched_pairs(candidates, left_df, right_df):
    match_pairs = candidates.index.to_frame(index=False)
    match_pairs.columns = ["left_index", "right_index"]

    merged = (
        match_pairs
        .merge(left_df, left_on="left_index", right_index=True, suffixes=('', '_left'))
        .merge(right_df, left_on="right_index", right_index=True, suffixes=('_left', '_right'))
    )
    return merged


compare = rl.Compare()
compare.string("ArtistLower", "ArtistLower", method="levenshtein", threshold=0.5, label="Artist_Sim")
compare.string("SongLower", "SongLower", method="levenshtein", threshold=0.6, label="Song_Sim")

features = compare.compute(pairs, df_spotify_remaining, df_million_song_remaining)

candidates = features[
    (features["Artist_Sim"] == 1) &
    (features["Song_Sim"] == 1)
]

df_approx_matches = get_matched_pairs(candidates, df_spotify_remaining, df_million_song_remaining)
print(len(df_approx_matches))

49


In [27]:
pd.set_option('display.max_rows', None)
df_approx_matches[['ArtistLower_right', 'ArtistLower_left', 'SongLower_right', 'SongLower_left']]

,ArtistLower_right,ArtistLower_left,SongLower_right,SongLower_left
0,eddie turner,eddie vedder,rise,rise
1,eddie turner,eddie neblett,rise,river
2,behemoth,behemoth,chant for eschaton 2000,chant for ezkaton 2000 e. v.
3,winds of plague,winds of plague,soldiers of doomsday,soldiers of doomsday
4,chris rea,chris rea,driving home for christmas,driving home for christmas - 2019 remaster
5,orbital,orbital,are we here ?,are we here?
6,dub pistols feat. tk & ashley slater,dub pistols;ashley slater,everyday stranger,everyday stranger
7,orbital,orbital,chime (edit),chime - edit
8,harry gregson-williams,harry gregson-williams,the ball,the battle
9,sean paul,sean paul;beyoncé,baby boy [feat. beyonce],baby boy (feat. beyoncé )


In [28]:
df_approx_matches = df_approx_matches.drop(index=[1,8,12,25,26,28,29,34,36,38,39,43,44,48])
# df_approx_matches = df_approx_matches.drop(index=[2, 9, 11, 12])
df_approx_matches[['ArtistLower_right', 'ArtistLower_left', 'SongLower_right', 'SongLower_left']]

,ArtistLower_right,ArtistLower_left,SongLower_right,SongLower_left
0,eddie turner,eddie vedder,rise,rise
2,behemoth,behemoth,chant for eschaton 2000,chant for ezkaton 2000 e. v.
3,winds of plague,winds of plague,soldiers of doomsday,soldiers of doomsday
4,chris rea,chris rea,driving home for christmas,driving home for christmas - 2019 remaster
5,orbital,orbital,are we here ?,are we here?
6,dub pistols feat. tk & ashley slater,dub pistols;ashley slater,everyday stranger,everyday stranger
7,orbital,orbital,chime (edit),chime - edit
9,sean paul,sean paul;beyoncé,baby boy [feat. beyonce],baby boy (feat. beyoncé )
10,shy fx & t power,shy fx;t. power,feelings,feelings
11,johnny osbourne,johnny osbourne,folly ranking,fally ranking


In [29]:
print(len(df_exact_join))
print(len(df_approx_matches))

100
35


In [31]:
df_exact_join.columns

Index(['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0_x', 'track_id', 'artists',
       'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit',
       'danceability', 'energy', 'key_x', 'loudness', 'mode_x', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'time_signature', 'track_genre', 'ArtistLower', 'SongLower',
       'Unnamed: 0_y', 'SongNumber', 'SongID', 'AlbumID', 'AlbumName',
       'ArtistID', 'ArtistName', 'Duration', 'KeySignature',
       'KeySignatureConfidence', 'Tempo', 'TimeSignature',
       'TimeSignatureConfidence', 'Title', 'Year', 'mbID', 'ArtistFamiliarity',
       'Hotness', 'end_of_fade_in', 'key_y', 'keyConfidence', 'Loudness',
       'mode_y', 'mode_confidence', 'start_of_fade_out'],
      dtype='object')

In [32]:
df_approx_matches['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0_x', 'track_id', 'artists',
       'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit',
       'danceability', 'energy', 'key_x', 'loudness', 'mode_x', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'time_signature', 'track_genre', 'ArtistLower', 'SongLower',
       'Unnamed: 0_y', 'SongNumber', 'SongID', 'AlbumID', 'AlbumName',
       'ArtistID', 'ArtistName', 'Duration', 'KeySignature',
       'KeySignatureConfidence', 'Tempo', 'TimeSignature',
       'TimeSignatureConfidence', 'Title', 'Year', 'mbID', 'ArtistFamiliarity',
       'Hotness', 'end_of_fade_in', 'key_y', 'keyConfidence', 'Loudness',
       'mode_y', 'mode_confidence', 'start_of_fade_out']

Index(['left_index', 'right_index', 'Unnamed: 0.2', 'Unnamed: 0.1',
       'Unnamed: 0_left', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
       'key_left', 'loudness', 'mode_left', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre', 'ArtistLower_left', 'SongLower_left', 'FirstWord_left',
       'Unnamed: 0_right', 'SongNumber', 'SongID', 'AlbumID', 'AlbumName',
       'ArtistID', 'ArtistName', 'Duration', 'KeySignature',
       'KeySignatureConfidence', 'Tempo', 'TimeSignature',
       'TimeSignatureConfidence', 'Title', 'Year', 'mbID', 'ArtistFamiliarity',
       'Hotness', 'end_of_fade_in', 'key_right', 'keyConfidence', 'Loudness',
       'mode_right', 'mode_confidence', 'start_of_fade_out',
       'ArtistLower_right', 'SongLower_right', 'FirstWord_right'],
      dtype='object')

In [ ]:
df_exact_join = df_exact_join[['artist', 'song', 'duration_ms', 'explicit', 'year','popularity', 'danceability', 'energy', 'key_x', 'loudness', 'mode_x','speechiness', 
                               'acousticness', 'instrumentalness', 'liveness','valence', 'tempo', 'genre', 'SongID', 'AlbumID','AlbumName', 'TimeSignature',
                               'TimeSignatureConfidence', 'Title', 'ArtistFamiliarity','Hotness', 'end_of_fade_in',  'start_of_fade_out']]
df_approx_matches = df_approx_matches[['artist', 'song', 'duration_ms', 'explicit', 'year','popularity', 'danceability', 'energy', 'key_left', 'loudness', 'mode_left','speechiness', 
                               'acousticness', 'instrumentalness', 'liveness','valence', 'tempo', 'genre', 'SongID', 'AlbumID','AlbumName', 'TimeSignature',
                               'TimeSignatureConfidence', 'Title', 'ArtistFamiliarity','Hotness', 'end_of_fade_in',  'start_of_fade_out']]

df_exact_join.columns = ['artist', 'song', 'duration_ms', 'explicit', 'year','popularity', 'danceability', 'energy', 'key', 'loudness', 'mode','speechiness', 
                               'acousticness', 'instrumentalness', 'liveness','valence', 'tempo', 'genre', 'SongID', 'AlbumID','AlbumName', 'TimeSignature',
                               'TimeSignatureConfidence', 'Title', 'ArtistFamiliarity','Hotness', 'end_of_fade_in',  'start_of_fade_out']
df_approx_matches.columns = ['artist', 'song', 'duration_ms', 'explicit', 'year','popularity', 'danceability', 'energy', 'key', 'loudness', 'mode','speechiness', 
                               'acousticness', 'instrumentalness', 'liveness','valence', 'tempo', 'genre', 'SongID', 'AlbumID','AlbumName', 'TimeSignature',
                               'TimeSignatureConfidence', 'Title', 'ArtistFamiliarity','Hotness', 'end_of_fade_in',  'start_of_fade_out']


In [30]:
df_exact_join = df_exact_join[['artist', 'song', 'duration_ms', 'explicit', 'year','popularity', 'danceability', 'energy', 'key_x', 'loudness', 'mode_x','speechiness', 
                               'acousticness', 'instrumentalness', 'liveness','valence', 'tempo', 'genre', 'SongID', 'AlbumID','AlbumName', 'TimeSignature',
                               'TimeSignatureConfidence', 'Title', 'ArtistFamiliarity','Hotness', 'end_of_fade_in',  'start_of_fade_out']]
df_approx_matches = df_approx_matches[['artist', 'song', 'duration_ms', 'explicit', 'year','popularity', 'danceability', 'energy', 'key_left', 'loudness', 'mode_left','speechiness', 
                               'acousticness', 'instrumentalness', 'liveness','valence', 'tempo', 'genre', 'SongID', 'AlbumID','AlbumName', 'TimeSignature',
                               'TimeSignatureConfidence', 'Title', 'ArtistFamiliarity','Hotness', 'end_of_fade_in',  'start_of_fade_out']]

df_exact_join.columns = ['artist', 'song', 'duration_ms', 'explicit', 'year','popularity', 'danceability', 'energy', 'key', 'loudness', 'mode','speechiness', 
                               'acousticness', 'instrumentalness', 'liveness','valence', 'tempo', 'genre', 'SongID', 'AlbumID','AlbumName', 'TimeSignature',
                               'TimeSignatureConfidence', 'Title', 'ArtistFamiliarity','Hotness', 'end_of_fade_in',  'start_of_fade_out']
df_approx_matches.columns = ['artist', 'song', 'duration_ms', 'explicit', 'year','popularity', 'danceability', 'energy', 'key', 'loudness', 'mode','speechiness', 
                               'acousticness', 'instrumentalness', 'liveness','valence', 'tempo', 'genre', 'SongID', 'AlbumID','AlbumName', 'TimeSignature',
                               'TimeSignatureConfidence', 'Title', 'ArtistFamiliarity','Hotness', 'end_of_fade_in',  'start_of_fade_out']





KeyError: "['artist', 'song', 'year', 'genre'] not in index"

In [11]:
df_integreated = pd.concat([df_exact_join, df_approx_matches], ignore_index=True)
print(len(df_integreated))


39


In [13]:
df_integreated.to_csv("../data/processed/Integrated.csv")